In [5]:
from google.colab import drive
import os, json, re, csv
from collections import Counter, defaultdict

drive.mount('/content/drive')

# ---- set these to your file on THIS account's Drive ----
SC_PATH    = '/content/drive/MyDrive/third_try/baseline__private__samples.jsonl'   # <- your file
OUT_CSV    = '/content/drive/MyDrive/third_try/submission.csv'           # <- output
ID_FIELD   = 'id'
TEXT_FIELD = 'samples'

def find_boxed(s):
    out, i = [], 0
    while True:
        j = s.find(r'\boxed{', i)
        if j < 0: break
        k = j + 7; depth = 1; start = k
        while k < len(s) and depth:
            depth += (s[k] == '{') - (s[k] == '}'); k += 1
        out.append(s[start:k-1]); i = k
    return out

def extract_answer(text):
    seg = text.rsplit('</think>', 1)[-1] if '</think>' in text else text
    boxes = find_boxed(seg) or find_boxed(text)
    return ", ".join(boxes) if boxes else None

def norm(a):
    if a is None: return None
    for t in (' ', '$', '\\left', '\\right', '\\!', '\\,'):
        a = a.replace(t, '')
    return a.strip().lower()

def get_text(x):                         # a sample may be a string or a dict
    if isinstance(x, str): return x
    if isinstance(x, dict):
        strs = [v for v in x.values() if isinstance(v, str)]
        return max(strs, key=len) if strs else ""   # the trace is the longest string
    return str(x)

by_id = defaultdict(list)
for line in open(SC_PATH):
    r = json.loads(line)
    for s in r[TEXT_FIELD]:              # the k generations for this question
        resp = get_text(s)
        by_id[r[ID_FIELD]].append((resp, norm(extract_answer(resp))))

final = {}
for rid, lst in by_id.items():
    votes = Counter(n for _, n in lst if n is not None)
    if votes:
        winner = votes.most_common(1)[0][0]
        final[rid] = next(resp for resp, n in lst if n == winner)  # a real trace giving the winner
    else:
        final[rid] = lst[0][0]           # nothing extractable; fall back to first sample

os.makedirs(os.path.dirname(OUT_CSV) or '.', exist_ok=True)   # don't die on a missing dir
with open(OUT_CSV, 'w', newline='') as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL)
    w.writerow(['id', 'response'])
    for rid in sorted(final):
        w.writerow([rid, final[rid]])

print(f'{len(final)} questions aggregated -> {OUT_CSV}')
ex = next(iter(by_id.values()))
print('sanity (one question vote spread):', Counter(n for _, n in ex if n))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
943 questions aggregated -> /content/drive/MyDrive/third_try/submission.csv
sanity (one question vote spread): Counter({'4,16,4,16': 6, '4,16': 1})


In [7]:
!pip install -q sympy numpy "antlr4-python3-runtime==4.11.1"

from google.colab import drive
import os, sys, json, csv, signal
from collections import defaultdict
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/third_try'                 # must hold judger.py + utils.py
SC_PATH     = f'{PROJECT_DIR}/baseline__private__samples.jsonl'
OUT_CSV     = f'{PROJECT_DIR}/submission.csv'
sys.path.insert(0, PROJECT_DIR)

from judger import Judger
J = Judger(strict_extract=False)
print('judger self-test  5==5:', J.is_equal("5","5"), '| 1/2==0.5:', J.is_equal("1/2","0.5"))
assert J.is_equal("5","5"), 'judger equality broken — check judger.py/utils.py + antlr4 install'

def get_text(x):
    if isinstance(x, str): return x
    if isinstance(x, dict):
        ss = [v for v in x.values() if isinstance(v, str)]
        return max(ss, key=len) if ss else ""
    return str(x)

def is_complete(t):                                    # clean boxed answer AFTER last </think>
    te = t.rfind('</think>')
    seg = t[te+8:] if te >= 0 else ''
    return bool(seg) and bool(J.extract_all_boxed(seg))

def answer_items(t):                                   # grader extraction (scans if truncated)
    raw = J.extract_ans(t)
    return [J.norm_ans_str(it) for it in J.split_by_comma(raw)] if raw else None

def items_equal(a, b):                                 # element-wise judger equivalence
    if a is None or b is None or len(a) != len(b): return False
    for x, y in zip(a, b):
        try:
            if not J.is_equal(x, y): return False
        except Exception:
            if x != y: return False
    return True

# show the sample structure once so nothing is assumed silently
with open(SC_PATH) as f:
    p = json.loads(f.readline())['samples'][0]
print('sample element:', type(p).__name__, ('keys '+str(list(p.keys())) if isinstance(p,dict) else ''))

final, n_q, n_samp, n_trunc, n_err = {}, 0, 0, 0, 0
for line in open(SC_PATH):
    r = json.loads(line); qid = r['id']
    parsed = []
    for s in r['samples']:
        t = get_text(s)
        if not t: continue
        n_samp += 1
        try:
            comp = is_complete(t); items = answer_items(t)
        except Exception:
            n_err += 1; comp, items = False, None
        if not comp: n_trunc += 1
        parsed.append((t, items, comp))

    # stage 1: cheap exact-key buckets (no sympy)
    buckets = {}
    for t, items, comp in parsed:
        key = tuple(items) if items is not None else None
        b = buckets.setdefault(key, {'count':0,'rep':items,'complete':None,'any':None})
        b['count'] += 1
        if b['any'] is None: b['any'] = t
        if comp and b['complete'] is None: b['complete'] = t

    votable = {k:v for k,v in buckets.items() if k is not None}
    if not votable:                                    # nothing extractable anywhere
        final[qid] = parsed[0][0] if parsed else ""; n_q += 1; continue

    # stage 2: merge judger-equivalent buckets (only a few distinct keys -> few is_equal calls)
    ks = list(votable); parent = {k:k for k in ks}
    def find(k):
        while parent[k] != k: parent[k] = parent[parent[k]]; k = parent[k]
        return k
    for i in range(len(ks)):
        for j in range(i+1, len(ks)):
            if find(ks[i]) != find(ks[j]) and items_equal(votable[ks[i]]['rep'], votable[ks[j]]['rep']):
                parent[find(ks[i])] = find(ks[j])

    groups = defaultdict(lambda: {'count':0,'complete':None,'any':None})
    for k in ks:
        g = groups[find(k)]; b = votable[k]
        g['count'] += b['count']
        if g['complete'] is None and b['complete']: g['complete'] = b['complete']
        if g['any'] is None: g['any'] = b['any']

    best = max(groups.values(), key=lambda g:(g['count'], g['complete'] is not None))
    final[qid] = best['complete'] or best['any']       # prefer a complete trace as the rep
    n_q += 1

os.makedirs(os.path.dirname(OUT_CSV) or '.', exist_ok=True)
with open(OUT_CSV, 'w', newline='') as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL); w.writerow(['id','response'])
    for qid in sorted(final): w.writerow([qid, final[qid]])

print(f'questions {n_q} | samples {n_samp} | truncated→scanned {n_trunc} | parse errors {n_err}')
print('wrote', OUT_CSV)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
judger self-test  5==5: True | 1/2==0.5: True
sample element: dict keys ['text', 'truncated', 'n_tok']
questions 943 | samples 6601 | truncated→scanned 196 | parse errors 0
wrote /content/drive/MyDrive/third_try/submission.csv
